In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/shristisaha903/dataset-4/Student folder/student_test/00000000/00000000.arcs
/kaggle/input/datasets/shristisaha903/dataset-4/Student folder/student_test/00000000/00000000.nodes
/kaggle/input/datasets/shristisaha903/dataset-4/Student folder/student_test/00000000/00000000.route
/kaggle/input/datasets/shristisaha903/dataset-4/Student folder/student_test/00000000/00000000.track
/kaggle/input/datasets/shristisaha903/dataset-4/Student folder/student_test/00000007/00000007.arcs
/kaggle/input/datasets/shristisaha903/dataset-4/Student folder/student_test/00000007/00000007.nodes
/kaggle/input/datasets/shristisaha903/dataset-4/Student folder/student_test/00000007/00000007.route
/kaggle/input/datasets/shristisaha903/dataset-4/Student folder/student_test/00000007/00000007.track
/kaggle/input/datasets/shristisaha903/dataset-4/Student folder/student_test/00000009/00000009.arcs
/kaggle/input/datasets/shristisaha903/dataset-4/Student folder/student_test/00000009/00000009.nodes
/ka

In [5]:
# ==========================================
# PATH SNAP: ADVANCED FEATURE ENGINEERING V2
# ==========================================
import os
import numpy as np
import pandas as pd
import time
import networkx as nx

# 1. DYNAMIC PATH FINDING
print("Locating dataset...")
track_files = []
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".track"):
            track_files.append(os.path.join(root, file))

if not track_files:
    raise Exception("No .track files found! Check if dataset is attached.")

sample_path = track_files[0]
BASE_DIR = sample_path.split("/Student folder/")[0]
print(f"Dataset found at: {BASE_DIR}")

# 2. HELPER FUNCTIONS
def load_track(path):
    return pd.read_csv(path, sep=r"\s+", header=None, names=["lon", "lat", "t"])

def load_nodes(path):
    df = pd.read_csv(path, sep=r"\s+", header=None, names=["lon", "lat"])
    df["node_id"] = df.index
    return df

def load_arcs(path):
    df = pd.read_csv(path, sep=r"\s+", header=None, names=["from_node", "to_node"])
    df["arc_id"] = df.index
    return df

def load_route(path):
    return pd.read_csv(path, sep=r"\s+", header=None, names=["arc_id"])

def get_candidates_and_meta(tr, nodes, arcs, max_candidates=3):
    lon0, lat0 = tr.lon.mean(), tr.lat.mean()
    px = (tr.lon.values - lon0) * 111320 * np.cos(np.radians(lat0))
    py = (tr.lat.values - lat0) * 111320
    nx_coords = (nodes.lon.values - lon0) * 111320 * np.cos(np.radians(lat0))
    ny_coords = (nodes.lat.values - lat0) * 111320
    
    from_x, from_y = nx_coords[arcs.from_node.values], ny_coords[arcs.from_node.values]
    to_x, to_y = nx_coords[arcs.to_node.values], ny_coords[arcs.to_node.values]
    
    abx, aby = to_x - from_x, to_y - from_y
    ab2 = abx**2 + aby**2
    ab2[ab2 == 0] = 1e-9 
    
    arc_lengths = np.sqrt(abx**2 + aby**2)
    arc_meta = pd.DataFrame({"arc_id": arcs.arc_id, "arc_length": arc_lengths})
    
    candidates = []
    for i in range(len(px)):
        apx, apy = px[i] - from_x, py[i] - from_y
        t = np.clip((apx * abx + apy * aby) / ab2, 0.0, 1.0)
        cx, cy = from_x + t * abx, from_y + t * aby
        dists = np.sqrt((px[i] - cx)**2 + (py[i] - cy)**2)
        
        closest_indices = np.argpartition(dists, max_candidates)[:max_candidates]
        closest_indices = closest_indices[np.argsort(dists[closest_indices])]
        
        for rank, idx in enumerate(closest_indices):
            dist_to_start = np.sqrt((px[i]-from_x[idx])**2 + (py[i]-from_y[idx])**2)
            dist_to_end = np.sqrt((px[i]-to_x[idx])**2 + (py[i]-to_y[idx])**2)
            
            candidates.append({
                'point_idx': i,
                'arc_id': arcs.arc_id.iloc[idx],
                'distance_m': dists[idx],
                'rank': rank,
                'dist_to_start_node': dist_to_start,
                'dist_to_end_node': dist_to_end
            })
    return pd.DataFrame(candidates), arc_meta

# 3. PROCESS FUNCTION WITH NEW FEATURES
def process_trip(trip_id, partition, base_dir):
    trip_id_str = str(trip_id).zfill(8)
    stem = os.path.join(base_dir, "Student folder", partition, trip_id_str, trip_id_str)
    
    if not os.path.exists(stem + ".track"):
        return None
    
    tr = load_track(stem + ".track")
    nodes = load_nodes(stem + ".nodes")
    arcs = load_arcs(stem + ".arcs")
    has_route = os.path.exists(stem + ".route")
    route = load_route(stem + ".route") if has_route else None
    
    # A. GPS Kinematics
    lon0, lat0 = tr.lon.mean(), tr.lat.mean()
    tr["x"] = (tr.lon.values - lon0) * 111320 * np.cos(np.radians(lat0))
    tr["y"] = (tr.lat.values - lat0) * 111320
    
    tr["dt"] = tr["t"].diff().fillna(1).clip(0, 3600)
    tr["dx"] = tr["x"].diff().fillna(0)
    tr["dy"] = tr["y"].diff().fillna(0)
    tr["delta_dist"] = np.sqrt(tr["dx"]**2 + tr["dy"]**2)
    tr["speed"] = (tr["delta_dist"] / tr["dt"].replace(0, np.nan)).fillna(0).clip(0, 50)
    tr["acceleration"] = tr["speed"].diff().fillna(0)
    tr["heading"] = (np.degrees(np.arctan2(tr["dx"], tr["dy"])) % 360).fillna(0)
    tr["heading_change"] = tr["heading"].diff().fillna(0)
    tr["heading_change"] = tr["heading_change"].apply(lambda x: min(x, 360 - x) if x > 180 else min(x, -360 - x) if x < -180 else x)
    tr["rolling_speed_mean"] = tr["speed"].rolling(window=5, min_periods=1).mean()
    tr["rolling_speed_std"] = tr["speed"].rolling(window=5, min_periods=1).std().fillna(0)
    tr["point_idx"] = tr.index
    tr["time_in_trip"] = tr["t"] - tr["t"].iloc[0]

    # B. Road Network Meta
    cand, arc_meta = get_candidates_and_meta(tr, nodes, arcs)
    nx_coords = (nodes.lon.values - lon0) * 111320 * np.cos(np.radians(lat0))
    ny_coords = (nodes.lat.values - lat0) * 111320
    from_x, from_y = nx_coords[arcs.from_node.values], ny_coords[arcs.from_node.values]
    to_x, to_y = nx_coords[arcs.to_node.values], ny_coords[arcs.to_node.values]
    road_heading = (np.degrees(np.arctan2(to_x - from_x, to_y - from_y)) % 360)
    arc_meta["road_heading"] = road_heading
    
    # NEW FEATURE 1: Arc Degree (Connectivity)
    G = nx.DiGraph()
    G.add_edges_from(zip(arcs.from_node, arcs.to_node))
    arc_meta["arc_degree"] = arcs.apply(lambda row: G.degree(row.from_node) + G.degree(row.to_node), axis=1)

    # C. Merge
    cand = cand.merge(arc_meta, on="arc_id", how="left")
    cand = cand.merge(tr[["point_idx", "speed", "acceleration", "heading", "heading_change", "dt", "rolling_speed_mean", "rolling_speed_std", "time_in_trip"]], on="point_idx", how="left")
    cand.rename(columns={"dt": "time_gap"}, inplace=True)

    # D. Advanced Candidate Features
    cand["heading_diff"] = np.abs(cand["heading"] - cand["road_heading"])
    cand["heading_diff"] = cand["heading_diff"].apply(lambda x: min(x, 360 - x))
    cand["dist_rank_0"] = cand.groupby("point_idx")["distance_m"].transform("min")
    cand["dist_rank_1"] = cand.groupby("point_idx")["distance_m"].transform(lambda x: x.nsmallest(2).max())
    cand["candidate_spread"] = cand["dist_rank_1"] - cand["dist_rank_0"]
    
    # NEW FEATURE 2: Implied Speed (Arc length / time gap)
    cand["implied_speed"] = cand["arc_length"] / cand["time_gap"].replace(0, 1)

    cand["next_arc_heading"] = cand.groupby("point_idx")["road_heading"].shift(-1).fillna(cand["road_heading"])
    cand["arc_turn_angle"] = np.abs(cand["road_heading"] - cand["next_arc_heading"])
    cand["arc_turn_angle"] = cand["arc_turn_angle"].apply(lambda x: min(x, 360 - x))

    # E. Sequence Features
    cand = cand.sort_values(by=["point_idx", "rank"])
    cand["prev_dist"] = cand.groupby("point_idx")["distance_m"].transform("first").shift(1).fillna(0)
    cand["prev_speed"] = cand.groupby("point_idx")["speed"].transform("first").shift(1).fillna(0)
    cand["time_since_prev"] = cand.groupby("point_idx")["time_gap"].transform("first").shift(1).fillna(0)
    
    # F. Labeling
    if has_route:
        gt_arcs = set(route["arc_id"].values)
        cand["target"] = cand["arc_id"].apply(lambda x: 1 if x in gt_arcs else 0)
    else:
        cand["target"] = np.nan
        
    cand["trip_id"] = trip_id
    return cand

# ==========================================
# 4. EXECUTE PIPELINE
# ==========================================
train_trips = []
for root, dirs, files in os.walk(os.path.join(BASE_DIR, "Student folder", "train")):
    for file in files:
        if file.endswith(".track"):
            train_trips.append(file.replace(".track", ""))

print(f"Found {len(train_trips)} training trips. Processing...")

all_trips = []
start_time = time.time()
for i, tid in enumerate(train_trips):
    if i % 10 == 0:
        print(f"  Processing {i}/{len(train_trips)}...")
    df = process_trip(tid, "train", BASE_DIR) 
    if df is not None:
        all_trips.append(df)

if not all_trips:
    raise Exception("Error: No trips were processed. Check path.")

final_df = pd.concat(all_trips, ignore_index=True).dropna()

print(f"\nTotal Features Engineered: {len(final_df.columns)}")
print(final_df.columns.tolist())

# Save the new V2 dataset
final_df.to_csv("advanced_features_full_v2.csv", index=False)
print(f"\nSaved {len(final_df)} rows to advanced_features_full_v2.csv")

Locating dataset...
Dataset found at: /kaggle/input/datasets/shristisaha903/dataset-4
Found 80 training trips. Processing...
  Processing 0/80...
  Processing 10/80...
  Processing 20/80...
  Processing 30/80...
  Processing 40/80...
  Processing 50/80...
  Processing 60/80...
  Processing 70/80...

Total Features Engineered: 29
['point_idx', 'arc_id', 'distance_m', 'rank', 'dist_to_start_node', 'dist_to_end_node', 'arc_length', 'road_heading', 'arc_degree', 'speed', 'acceleration', 'heading', 'heading_change', 'time_gap', 'rolling_speed_mean', 'rolling_speed_std', 'time_in_trip', 'heading_diff', 'dist_rank_0', 'dist_rank_1', 'candidate_spread', 'implied_speed', 'next_arc_heading', 'arc_turn_angle', 'prev_dist', 'prev_speed', 'time_since_prev', 'target', 'trip_id']

Saved 625251 rows to advanced_features_full_v2.csv
